<a href="https://colab.research.google.com/github/Prajwaln08/SQLSmith/blob/main/01_SQLSmith_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQLSmith Milestone A: Notebook 01, Baseline Evaluation

This notebook builds the baseline for the SQLSmith text-to-SQL project:

1. Environment setup on a Colab T4 GPU
2. Qwen2.5-Coder-1.5B-Instruct loaded in 4-bit
3. DuckDB demo database with a 12 question evaluation
4. Spider 1.0 dev set evaluation on 50 questions
5. First pass error analysis of the failed questions

All results are saved to the shared project folder:
`/content/drive/MyDrive/Prajwal/Projects/SQLSmith/results`

## 1. GPU and Environment Check

Confirms the runtime has a GPU attached and prints the device name and PyTorch version. The notebook stops here if no GPU is available.

In [1]:
!nvidia-smi

import torch

# Fail fast if the notebook was started without a GPU runtime
assert torch.cuda.is_available(), "GPU is not enabled"

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

Sun Nov 22 18:11:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Dependencies

Installs model loading, quantization, SQL parsing and data handling libraries. The pip dependency resolver warnings about pandas and sqlglot versions are expected on Colab and safe to ignore.

In [2]:
# Core dependencies: transformers for the model, bitsandbytes for 4-bit
# quantization, duckdb and sqlglot for SQL execution and validation
%pip install -q -U transformers accelerate \
    bitsandbytes duckdb sqlglot pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
ibis-framework 9.5.0 requires sqlglot<25.21,>=23.4, but you have sqlglot 30.33.0 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.


## 3. Mount Google Drive and Set the Results Directory

Mounts Drive once at the start and defines `RESULTS_DIR`, the single project results folder. Every save in this notebook, and in notebooks 02 to 05, must write to `RESULTS_DIR` so nothing is lost when the Colab runtime disconnects.

In [3]:
from google.colab import drive
from pathlib import Path

# Mount Drive early so every save below lands in the project folder
drive.mount("/content/drive")

# Single source of truth for the project results folder.
# Use this exact variable in every notebook (01 to 05).
RESULTS_DIR = Path(
    "/content/drive/MyDrive/Prajwal/Projects/SQLSmith/results"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Saving results to:", RESULTS_DIR)

Mounted at /content/drive
Saving results to: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/results


## 4. Load Qwen2.5-Coder-1.5B-Instruct in 4-bit

Loads the tokenizer and the model with NF4 quantization so it fits comfortably in the T4 GPU memory. The model is set to evaluation mode since this notebook only runs inference.

In [4]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

# 4-bit NF4 quantization keeps the model small enough for a T4 GPU
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto"
)

# Inference only, no training in this notebook
model.eval()

print("Model loaded successfully!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


## 5. Create the Demo DuckDB Database

Builds a small in-memory `sales` table with 5 rows. This toy database is used for the first smoke test and the 12 question demo evaluation before moving to Spider.

In [5]:
import duckdb

# In-memory database, recreated every time this cell runs
con = duckdb.connect(":memory:")

con.execute("""
CREATE TABLE sales (
    order_id INTEGER,
    region VARCHAR,
    revenue DOUBLE,
    order_date DATE
)
""")

con.execute("""
INSERT INTO sales VALUES
(1, 'North', 1200, '2026-01-05'),
(2, 'South', 2500, '2026-01-07'),
(3, 'North', 1800, '2026-01-12'),
(4, 'West', 3200, '2026-01-14'),
(5, 'South', 1500, '2026-01-18')
""")

print(con.execute("SELECT * FROM sales").df())

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


   order_id region  revenue order_date
0         1  North   1200.0 2026-01-05
1         2  South   2500.0 2026-01-07
2         3  North   1800.0 2026-01-12
3         4   West   3200.0 2026-01-14
4         5  South   1500.0 2026-01-18


## 6. Single Question Smoke Test

Runs one question end to end: build the chat prompt, generate with greedy decoding, and decode only the newly generated tokens into SQL.

In [6]:
import torch

question = "What is the total revenue for each region?"

schema = """
Table: sales
Columns:
- order_id INTEGER
- region VARCHAR
- revenue DOUBLE
- order_date DATE
"""

messages = [
    {
        "role": "system",
        "content": (
            "You are an expert SQL assistant. "
            "Generate exactly one DuckDB SELECT query. "
            "Use only the provided schema. "
            "Return SQL only, without markdown."
        )
    },
    {
        "role": "user",
        "content": f"Schema:\n{schema}\n\nQuestion: {question}"
    }
]

# Step 1: Create chat prompt as text
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Step 2: Convert prompt to tensors
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False
)

# Step 3: Explicitly extract input IDs
input_ids = inputs["input_ids"].to(model.device)
attention_mask = inputs["attention_mask"].to(model.device)

print("Input tensor shape:", input_ids.shape)

# Step 4: Generate SQL with greedy decoding for reproducibility
with torch.inference_mode():
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

# Step 5: Decode only the newly generated tokens
generated_tokens = outputs[0, input_ids.shape[-1]:]

generated_sql = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

print("\nGenerated SQL:")
print(generated_sql)

Input tensor shape: torch.Size([1, 78])

Generated SQL:
SELECT region, SUM(revenue) AS total_revenue FROM sales GROUP BY region


## 7. Validate and Execute the Generated SQL

Parses the generated SQL with sqlglot, checks that it is a single SELECT on the known table, blocks write operations, then executes it against DuckDB.

In [7]:
import sqlglot
from sqlglot import exp

statements = sqlglot.parse(
    generated_sql,
    read="duckdb"
)

# Exactly one statement, and it must be a SELECT
assert len(statements) == 1
assert isinstance(statements[0], exp.Select)

# Restrict this tiny demo to its one known table.
tables = {
    t.name for t in statements[0].find_all(exp.Table)
}
assert tables == {"sales"}

# Block non-query operations inside the parsed tree.
blocked = (
    exp.Insert, exp.Update, exp.Delete,
    exp.Create, exp.Drop, exp.Alter,
    exp.Command
)

assert not any(
    isinstance(node, blocked)
    for node in statements[0].walk()
)

result = con.execute(generated_sql).df()

display(result)

,region,total_revenue
0,North,3000.0
1,South,4000.0
2,West,3200.0


## 8. Define the 12 Demo Test Cases

Hand-written natural language questions paired with gold SQL for the demo sales table. These cover aggregation, filtering, grouping, ordering and date logic.

In [8]:
# 12 question and gold SQL pairs for the demo sales table
test_cases = [
    {
        "id": 1,
        "question": "What is the total revenue for each region?",
        "gold_sql": """
            SELECT region, SUM(revenue)
            FROM sales GROUP BY region
        """
    },
    {
        "id": 2,
        "question": "What is the total revenue across all orders?",
        "gold_sql": "SELECT SUM(revenue) FROM sales"
    },
    {
        "id": 3,
        "question": "How many orders are there?",
        "gold_sql": "SELECT COUNT(*) FROM sales"
    },
    {
        "id": 4,
        "question": "What is the average order revenue?",
        "gold_sql": "SELECT AVG(revenue) FROM sales"
    },
    {
        "id": 5,
        "question": "Which region has the highest total revenue?",
        "gold_sql": """
            SELECT region FROM sales
            GROUP BY region
            ORDER BY SUM(revenue) DESC
            LIMIT 1
        """
    },
    {
        "id": 6,
        "question": "What is the maximum revenue of a single order?",
        "gold_sql": "SELECT MAX(revenue) FROM sales"
    },
    {
        "id": 7,
        "question": "How many orders came from the South region?",
        "gold_sql": """
            SELECT COUNT(*) FROM sales
            WHERE region = 'South'
        """
    },
    {
        "id": 8,
        "question": "What is the total revenue from North?",
        "gold_sql": """
            SELECT SUM(revenue) FROM sales
            WHERE region = 'North'
        """
    },
    {
        "id": 9,
        "question": "Show the order IDs for orders with revenue above 2000.",
        "gold_sql": """
            SELECT order_id FROM sales
            WHERE revenue > 2000
        """
    },
    {
        "id": 10,
        "question": "What is the total revenue in January 2026?",
        "gold_sql": """
            SELECT SUM(revenue) FROM sales
            WHERE order_date >= DATE '2026-01-01'
              AND order_date < DATE '2026-02-01'
        """
    },
    {
        "id": 11,
        "question": "How many distinct regions have sales?",
        "gold_sql": """
            SELECT COUNT(DISTINCT region) FROM sales
        """
    },
    {
        "id": 12,
        "question": "What is the total revenue for orders after January 10, 2026?",
        "gold_sql": """
            SELECT SUM(revenue) FROM sales
            WHERE order_date > DATE '2026-01-10'
        """
    }
]

print(f"Loaded {len(test_cases)} test cases")

Loaded 12 test cases


## 9. SQL Generation Helper

Wraps prompt building, generation and decoding into one function that also measures per question generation latency and output token count.

In [9]:
import time
import torch

def generate_sql(question, schema):
    """Generate one DuckDB SELECT query and measure generation latency."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are a text-to-SQL assistant. "
                "Generate exactly one DuckDB SELECT query. "
                "Use only the provided schema. "
                "Return SQL only. No markdown or explanation."
            )
        },
        {
            "role": "user",
            "content": (
                f"Database schema:\n{schema}\n\n"
                f"Question: {question}"
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    )

    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    # Synchronize before and after generation for accurate GPU timing
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    latency = time.perf_counter() - start

    new_tokens = outputs[0, input_ids.shape[-1]:]

    sql = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return sql, latency, len(new_tokens)

## 10. Validation and Result Comparison Helpers

Three helpers: `validate_demo_sql` whitelists SQL constructs, tables and columns; `normalize_result` makes row sets comparable; `compare_execution` runs predicted and gold SQL and compares their results.

In [10]:
import sqlglot
from sqlglot import exp
from collections import Counter
import pandas as pd
import math
import re

def validate_demo_sql(sql):
    """Allow only a safe whitelist of SELECT constructs on the sales table."""
    if "```" in sql:
        raise ValueError("Model returned markdown instead of SQL")

    statements = sqlglot.parse(sql, read="duckdb")

    if len(statements) != 1:
        raise ValueError("Expected exactly one SQL statement")

    tree = statements[0]

    if not isinstance(tree, exp.Select):
        raise ValueError("Only SELECT is supported")

    # Restrict the syntax of this toy example.
    allowed_nodes = {
        exp.Select, exp.From, exp.Table, exp.Column,
        exp.Identifier, exp.Literal, exp.Alias,
        exp.Star, exp.Where, exp.Group, exp.Order,
        exp.Ordered, exp.Limit, exp.Count,
        exp.Sum, exp.Avg, exp.Max, exp.Min,
        exp.Distinct, exp.EQ, exp.GT, exp.GTE,
        exp.LT, exp.LTE, exp.NEQ, exp.And,
        exp.Or, exp.Paren, exp.Add, exp.Sub,
        exp.Mul, exp.Div, exp.Cast, exp.DataType,
        exp.Date, exp.Between, exp.In,
        exp.Is, exp.Null, exp.Not
    }

    for node in tree.walk():
        if type(node) not in allowed_nodes:
            raise ValueError(
                f"Unsupported SQL construct: {type(node).__name__}"
            )

    tables = {t.name.lower() for t in tree.find_all(exp.Table)}

    if tables != {"sales"}:
        raise ValueError("Unauthorized table reference")

    columns = {c.name.lower() for c in tree.find_all(exp.Column)}

    allowed_columns = {
        "order_id", "region", "revenue", "order_date"
    }

    # Output aliases are not included as physical table columns.
    aliases = {
        a.alias.lower()
        for a in tree.find_all(exp.Alias)
        if a.alias
    }

    if not columns.issubset(allowed_columns | aliases):
        raise ValueError("Unknown column reference")

    return True


def normalize_result(rows):
    """Convert result rows into a Counter so row order does not matter."""
    def normalize_value(v):
        if v is None:
            return None
        if isinstance(v, float):
            if math.isnan(v):
                return "NaN"
            return round(v, 6)
        return str(v) if not isinstance(v, (int, float)) else v

    return Counter(
        tuple(normalize_value(v) for v in row)
        for row in rows
    )


def compare_execution(predicted_sql, gold_sql):
    """Execute both queries and compare normalized results."""
    try:
        validate_demo_sql(predicted_sql)

        predicted = con.execute(predicted_sql).fetchall()
        expected = con.execute(gold_sql).fetchall()

        return (
            normalize_result(predicted)
            == normalize_result(expected),
            None
        )

    except Exception as e:
        return False, str(e)

## 11. Run the 12 Question Demo Evaluation

Generates SQL for each test case, compares execution results against the gold SQL, and reports execution accuracy and latency statistics.

In [11]:
# Evaluate every demo test case and collect per question metrics
results = []

for case in test_cases:
    print(f"Evaluating question {case['id']}...")

    generated_sql, latency, tokens = generate_sql(
        case["question"],
        schema
    )

    correct, error = compare_execution(
        generated_sql,
        case["gold_sql"]
    )

    results.append({
        "question_id": case["id"],
        "question": case["question"],
        "generated_sql": generated_sql,
        "gold_sql": case["gold_sql"],
        "correct": correct,
        "latency_seconds": round(latency, 3),
        "output_tokens": tokens,
        "error": error
    })

df = pd.DataFrame(results)

display(df[[
    "question_id",
    "correct",
    "latency_seconds",
    "output_tokens",
    "error"
]])

accuracy = df["correct"].mean() * 100

print(f"\nExecution accuracy: {accuracy:.2f}%")
print(f"Average generation latency: {df['latency_seconds'].mean():.2f}s")
print(f"p95 generation latency: {df['latency_seconds'].quantile(0.95):.2f}s")

Evaluating question 1...
Evaluating question 2...
Evaluating question 3...
Evaluating question 4...
Evaluating question 5...
Evaluating question 6...
Evaluating question 7...
Evaluating question 8...
Evaluating question 9...
Evaluating question 10...
Evaluating question 11...
Evaluating question 12...


,question_id,correct,latency_seconds,output_tokens,error
0,1,True,1.622,17,NaN
1,2,True,0.955,8,NaN
2,3,True,0.734,6,NaN
3,4,True,1.033,8,NaN
4,5,False,6.519,31,NaN
5,6,True,0.733,8,NaN
6,7,True,0.935,14,NaN
7,8,True,0.940,14,NaN
8,9,True,0.946,14,NaN
9,10,False,2.375,36,Unsupported SQL construct: Extract



Execution accuracy: 83.33%
Average generation latency: 1.59s
p95 generation latency: 4.24s


## 12. Save Demo Results to Google Drive

Writes the per question results and the summary metrics to `RESULTS_DIR`. This previously saved to a local `results/` folder on the runtime, which was lost on disconnect; it now goes to Drive.

In [12]:
import json

# Per question results for the 12 question demo
df.to_csv(
    RESULTS_DIR / "baseline_1_5b.csv",
    index=False
)

# Summary metrics for the demo run
summary = {
    "model": MODEL_ID,
    "experiment": "baseline_1_5b",
    "num_questions": len(df),
    "correct_queries": int(df["correct"].sum()),
    "execution_accuracy": float(df["correct"].mean()),
    "mean_latency_seconds": float(df["latency_seconds"].mean()),
    "p95_latency_seconds": float(df["latency_seconds"].quantile(0.95))
}

with open(RESULTS_DIR / "baseline_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("Demo results saved to:", RESULTS_DIR)

{
  "model": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
  "experiment": "baseline_1_5b",
  "num_questions": 12,
  "correct_queries": 10,
  "execution_accuracy": 0.8333333333333334,
  "mean_latency_seconds": 1.5942499999999997,
  "p95_latency_seconds": 4.239799999999997
}
Demo results saved to: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/results


## 13. Download the Spider Dataset

Downloads the Spider 1.0 archive from Hugging Face, verifies its SHA-256 checksum, and extracts it to the local runtime at `/content/sqlsmith_data`.

In [13]:
%pip install -q huggingface_hub

from huggingface_hub import hf_hub_download
from pathlib import Path
import hashlib
import zipfile
import json

zip_path = hf_hub_download(
    repo_id="HAL-9001/spider-databases",
    repo_type="dataset",
    filename="spider_data.zip"
)

# Verify the archive before extracting anything from it
expected_hash = (
    "00636695dabed6b5f4b8328a16b13e069a2f16591d5efcce57660669c85b121b"
)

actual_hash = hashlib.sha256(
    Path(zip_path).read_bytes()
).hexdigest()

assert actual_hash == expected_hash, "Dataset checksum mismatch"

# Extract to the local runtime; the dataset is re-downloaded per session
output_dir = Path("/content/sqlsmith_data")
output_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path) as z:
    z.extractall(output_dir)

print("Spider downloaded and extracted")

spider_data.zip: reconstructing file:   0%|          |  0.00B /  206MB            

spider_data.zip: downloading bytes:           |  0.00B            

Spider downloaded and extracted


## 14. Load Spider Questions and Schemas

Loads the dev set questions, the table metadata, and checks that the SQLite database folder is present. Builds a `db_id` to schema lookup.

In [14]:
from pathlib import Path
import json

root = Path("/content/sqlsmith_data/spider_data")

dev_path = root / "dev.json"
tables_path = root / "tables.json"
db_dir = root / "database"

# Confirm the extraction produced the expected files
assert dev_path.exists(), "dev.json missing"
assert tables_path.exists(), "tables.json missing"
assert db_dir.exists(), "SQLite databases missing"

with open(dev_path) as f:
    dev_examples = json.load(f)

with open(tables_path) as f:
    table_metadata = json.load(f)

# Lookup table: db_id to its schema metadata
schemas = {
    item["db_id"]: item
    for item in table_metadata
}

print("Evaluation questions:", len(dev_examples))
print("Database schemas:", len(schemas))
print("First question:", dev_examples[0]["question"])

Evaluation questions: 1034
Database schemas: 166
First question: How many singers do we have?


## 15. Spider Schema Formatting Helper

Turns Spider schema metadata into the plain text format given to the model: one block per table with typed columns, followed by the foreign keys.

In [15]:
def get_spider_schema(db_id):
    """Format a Spider database schema as plain text for the prompt."""
    metadata = schemas[db_id]

    tables = metadata["table_names_original"]
    columns = metadata["column_names_original"]
    types = metadata["column_types"]

    output = []

    for table_index, table_name in enumerate(tables):
        table_columns = []

        for col_index, (idx, col_name) in enumerate(columns):
            if idx == table_index:
                col_type = types[col_index]
                table_columns.append(
                    f"  {col_name} {col_type}"
                )

        output.append(
            f"Table: {table_name}\n"
            + "\n".join(table_columns)
        )

    output.append("\nForeign keys:")

    for col_a, col_b in metadata["foreign_keys"]:
        ta, ca = columns[col_a]
        tb, cb = columns[col_b]

        if ta >= 0 and tb >= 0:
            output.append(
                f"{tables[ta]}.{ca} = {tables[tb]}.{cb}"
            )

    return "\n\n".join(output)


# Sanity check on the first dev example
example = dev_examples[0]

print(get_spider_schema(example["db_id"]))

Table: stadium
  Stadium_ID number
  Location text
  Name text
  Capacity number
  Highest number
  Lowest number
  Average number

Table: singer
  Singer_ID number
  Name text
  Country text
  Song_Name text
  Song_release_year text
  Age number
  Is_male others

Table: concert
  concert_ID number
  concert_Name text
  Theme text
  Stadium_ID text
  Year text

Table: singer_in_concert
  concert_ID number
  Singer_ID text


Foreign keys:

concert.Stadium_ID = stadium.Stadium_ID

singer_in_concert.Singer_ID = singer.Singer_ID

singer_in_concert.concert_ID = concert.concert_ID


## 16. Spider SQL Generation Helper

Same generation flow as the demo helper, but prompts for SQLite instead of DuckDB and allows more output tokens for the harder Spider questions.

In [16]:
import torch
import time

def generate_spider_sql(question, schema_text):
    """Generate one SQLite SELECT query for a Spider question."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert text-to-SQL assistant. "
                "Generate exactly one valid SQLite SELECT query. "
                "Use only the tables and columns in the schema. "
                "Return SQL only, without markdown or explanation."
            )
        },
        {
            "role": "user",
            "content": (
                f"Schema:\n{schema_text}\n\n"
                f"Question: {question}"
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    )

    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    # Synchronize before and after generation for accurate GPU timing
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    latency = time.perf_counter() - start

    generated = tokenizer.decode(
        outputs[0, input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    return generated, latency

## 17. Safe SQLite Execution Helpers

`execute_spider_sql` runs a single SELECT against the right Spider database in read-only mode with a 5 second timeout. `normalize_rows` makes result sets comparable regardless of row order.

In [17]:
import sqlite3
import sqlglot
from sqlglot import exp
from collections import Counter

def execute_spider_sql(db_id, sql):
    """Run one SELECT against a Spider SQLite database, read-only, 5s limit."""
    db_path = (db_dir / db_id / f"{db_id}.sqlite").resolve()

    if not db_path.is_file():
        raise FileNotFoundError(db_path)

    parsed = sqlglot.parse(sql, read="sqlite")

    if len(parsed) != 1 or not isinstance(parsed[0], exp.Select):
        raise ValueError("Expected a single SELECT statement")

    # This benchmark sandbox is not a production SQL security boundary.
    uri = f"{db_path.as_uri()}?mode=ro"

    connection = sqlite3.connect(uri, uri=True, timeout=5)

    try:
        connection.execute("PRAGMA query_only = ON")

        # Abort queries that run longer than 5 seconds
        deadline = time.monotonic() + 5

        def limit_execution():
            return int(time.monotonic() > deadline)

        connection.set_progress_handler(limit_execution, 1000)

        cursor = connection.execute(sql)
        return cursor.fetchall()

    finally:
        connection.close()


def normalize_rows(rows):
    """Convert result rows into a Counter so row order does not matter."""
    return Counter(
        tuple(
            round(v, 6) if isinstance(v, float) else v
            for v in row
        )
        for row in rows
    )

## 18. Run the 50 Question Spider Evaluation

Evaluates the first 50 dev questions: generate SQL, execute both predicted and gold queries, and compare results. Failures are recorded with their error message instead of stopping the loop.

In [18]:
import pandas as pd

# Freeze the sample selection for reproducibility.
sample = dev_examples[:50]

spider_results = []

for index, example in enumerate(sample, start=1):
    db_id = example["db_id"]
    question = example["question"]
    gold_sql = example["query"]

    # Defaults in case any step below fails
    prediction = ""
    latency = None
    correct = False
    error = None

    try:
        schema_text = get_spider_schema(db_id)

        prediction, latency = generate_spider_sql(
            question,
            schema_text
        )

        pred_rows = execute_spider_sql(db_id, prediction)
        gold_rows = execute_spider_sql(db_id, gold_sql)

        correct = (
            normalize_rows(pred_rows)
            == normalize_rows(gold_rows)
        )

    except Exception as e:
        error = str(e)

    spider_results.append({
        "index": index,
        "db_id": db_id,
        "question": question,
        "gold_sql": gold_sql,
        "predicted_sql": prediction,
        "correct": correct,
        "latency_seconds": latency,
        "error": error
    })

    print(
        f"[{index}/{len(sample)}] "
        f"{'PASS' if correct else 'FAIL'}"
    )

spider_df = pd.DataFrame(spider_results)

display(spider_df[[
    "index", "db_id", "correct",
    "latency_seconds", "error"
]])

print(
    "Provisional result-match accuracy:",
    f"{spider_df['correct'].mean() * 100:.2f}%"
)

print(
    "Mean generation latency:",
    spider_df["latency_seconds"].mean()
)

[1/50] PASS
[2/50] PASS
[3/50] PASS
[4/50] PASS
[5/50] PASS
[6/50] PASS
[7/50] FAIL
[8/50] FAIL
[9/50] FAIL
[10/50] PASS
[11/50] FAIL
[12/50] FAIL
[13/50] FAIL
[14/50] FAIL
[15/50] PASS
[16/50] PASS
[17/50] FAIL
[18/50] PASS
[19/50] FAIL
[20/50] FAIL
[21/50] PASS
[22/50] PASS
[23/50] FAIL
[24/50] PASS
[25/50] PASS
[26/50] PASS
[27/50] PASS
[28/50] PASS
[29/50] FAIL
[30/50] FAIL
[31/50] FAIL
[32/50] FAIL
[33/50] FAIL
[34/50] PASS
[35/50] FAIL
[36/50] PASS
[37/50] PASS
[38/50] FAIL
[39/50] FAIL
[40/50] FAIL
[41/50] FAIL
[42/50] FAIL
[43/50] FAIL
[44/50] PASS
[45/50] PASS
[46/50] PASS
[47/50] PASS
[48/50] FAIL
[49/50] FAIL
[50/50] FAIL


,index,db_id,correct,latency_seconds,error
0,1,concert_singer,True,1.379505,NaN
1,2,concert_singer,True,1.143007,NaN
2,3,concert_singer,True,2.068415,NaN
3,4,concert_singer,True,2.294654,NaN
4,5,concert_singer,True,5.696822,NaN
5,6,concert_singer,True,5.703762,NaN
6,7,concert_singer,False,15.649143,no such column: T2.song_name
7,8,concert_singer,False,9.554916,no such column: T2.song_name
8,9,concert_singer,False,5.933174,NaN
9,10,concert_singer,True,1.468719,NaN


Provisional result-match accuracy: 48.00%
Mean generation latency: 3.628774267359997


## 19. Save Spider Results to Google Drive

Writes the 50 question results and the summary metrics to `RESULTS_DIR` in the shared project folder.

In [19]:
import json

# Per question results for the 50 question Spider run
spider_df.to_csv(
    RESULTS_DIR / "spider_1_5b_baseline_50.csv",
    index=False
)

# Summary metrics for the Spider run
summary = {
    "model": "Qwen2.5-Coder-1.5B-Instruct",
    "dataset": "Spider 1.0 dev",
    "num_questions": len(spider_df),
    "num_correct": int(spider_df["correct"].sum()),
    "provisional_result_accuracy": float(
        spider_df["correct"].mean()
    ),
    "mean_generation_latency_seconds": float(
        spider_df["latency_seconds"].mean()
    )
}

with open(RESULTS_DIR / "spider_baseline_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("Spider results saved to:", RESULTS_DIR)

{
  "model": "Qwen2.5-Coder-1.5B-Instruct",
  "dataset": "Spider 1.0 dev",
  "num_questions": 50,
  "num_correct": 24,
  "provisional_result_accuracy": 0.48,
  "mean_generation_latency_seconds": 3.628774267359997
}
Spider results saved to: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/results


## 20. Inspect Failed Questions

Filters the Spider results down to the failed questions and displays them with their gold SQL, predicted SQL and error message.

In [20]:
import pandas as pd

# Keep only the questions that did not match the gold results
failures = spider_df[
    spider_df["correct"] == False
].copy()

print("Total evaluated:", len(spider_df))
print("Correct:", int(spider_df["correct"].sum()))
print("Failures:", len(failures))

display(
    failures[
        [
            "index",
            "db_id",
            "question",
            "gold_sql",
            "predicted_sql",
            "error"
        ]
    ]
)

Total evaluated: 50
Correct: 24
Failures: 26


,index,db_id,question,gold_sql,predicted_sql,error
6,7,concert_singer,Show the name and the release year of the song...,"SELECT song_name , song_release_year FROM sin...","SELECT T2.song_name, T2.song_release_year FROM...",no such column: T2.song_name
7,8,concert_singer,What are the names and release years for all t...,"SELECT song_name , song_release_year FROM sin...","SELECT T2.song_name, T2.song_release_year FROM...",no such column: T2.song_name
8,9,concert_singer,What are all distinct countries where singers ...,SELECT DISTINCT country FROM singer WHERE age ...,SELECT DISTINCT T1.Country FROM singer AS T1 I...,NaN
10,11,concert_singer,Show all countries and the number of singers i...,"SELECT country , count(*) FROM singer GROUP B...","SELECT T1.Country, COUNT(T2.Singer_ID) FROM si...",NaN
11,12,concert_singer,How many singers are from each country?,"SELECT country , count(*) FROM singer GROUP B...","SELECT COUNT(*) , Country FROM singer GROUP B...",NaN
12,13,concert_singer,List all song names by singers above the avera...,SELECT song_name FROM singer WHERE age > (SE...,SELECT T2.song_name FROM singer AS T1 JOIN sin...,no such column: T2.song_name
13,14,concert_singer,What are all the song names by singers who are...,SELECT song_name FROM singer WHERE age > (SE...,SELECT T2.song_name FROM singer AS T1 INNER JO...,no such column: T2.song_name
16,17,concert_singer,What is the maximum capacity and the average o...,"select max(capacity), average from stadium","SELECT max(capacity), avg(capacity) FROM stadium",NaN
18,19,concert_singer,What is the name and capacity for the stadium ...,"SELECT name , capacity FROM stadium ORDER BY ...","SELECT T1.Name, T1.Capacity FROM stadium AS T1...",no such column: T3.Age
19,20,concert_singer,What is the name and capacity for the stadium ...,"SELECT name , capacity FROM stadium ORDER BY ...","SELECT T1.Name, T1.Capacity FROM stadium AS T1...",no such column: Average_number


## 21. First Pass Error Classification

Buckets each failure by its error message: hallucinated columns or tables, syntax errors, timeouts, other execution errors, or a semantic mismatch that needs manual review in notebook 02.

In [21]:
def classify_initial_error(row):
    """Assign a coarse error category based on the recorded error message."""
    error = row["error"]

    if pd.notna(error) and str(error).strip():
        error_text = str(error).lower()

        if "no such column" in error_text:
            return "Hallucinated / invalid column"

        if "no such table" in error_text:
            return "Hallucinated / invalid table"

        if "syntax error" in error_text:
            return "SQL syntax error"

        if "interrupted" in error_text:
            return "Execution timeout"

        return "Other execution error"

    # Query executed but returned different results than the gold SQL
    return "Semantic mismatch - manual review"


failures["error_category"] = failures.apply(
    classify_initial_error,
    axis=1
)

print(failures["error_category"].value_counts())

display(
    failures[
        [
            "index",
            "question",
            "predicted_sql",
            "gold_sql",
            "error_category"
        ]
    ]
)

error_category
Hallucinated / invalid column        12
Semantic mismatch - manual review     9
Other execution error                 5
Name: count, dtype: int64


,index,question,predicted_sql,gold_sql,error_category
6,7,Show the name and the release year of the song...,"SELECT T2.song_name, T2.song_release_year FROM...","SELECT song_name , song_release_year FROM sin...",Hallucinated / invalid column
7,8,What are the names and release years for all t...,"SELECT T2.song_name, T2.song_release_year FROM...","SELECT song_name , song_release_year FROM sin...",Hallucinated / invalid column
8,9,What are all distinct countries where singers ...,SELECT DISTINCT T1.Country FROM singer AS T1 I...,SELECT DISTINCT country FROM singer WHERE age ...,Semantic mismatch - manual review
10,11,Show all countries and the number of singers i...,"SELECT T1.Country, COUNT(T2.Singer_ID) FROM si...","SELECT country , count(*) FROM singer GROUP B...",Semantic mismatch - manual review
11,12,How many singers are from each country?,"SELECT COUNT(*) , Country FROM singer GROUP B...","SELECT country , count(*) FROM singer GROUP B...",Semantic mismatch - manual review
12,13,List all song names by singers above the avera...,SELECT T2.song_name FROM singer AS T1 JOIN sin...,SELECT song_name FROM singer WHERE age > (SE...,Hallucinated / invalid column
13,14,What are all the song names by singers who are...,SELECT T2.song_name FROM singer AS T1 INNER JO...,SELECT song_name FROM singer WHERE age > (SE...,Hallucinated / invalid column
16,17,What is the maximum capacity and the average o...,"SELECT max(capacity), avg(capacity) FROM stadium","select max(capacity), average from stadium",Semantic mismatch - manual review
18,19,What is the name and capacity for the stadium ...,"SELECT T1.Name, T1.Capacity FROM stadium AS T1...","SELECT name , capacity FROM stadium ORDER BY ...",Hallucinated / invalid column
19,20,What is the name and capacity for the stadium ...,"SELECT T1.Name, T1.Capacity FROM stadium AS T1...","SELECT name , capacity FROM stadium ORDER BY ...",Hallucinated / invalid column


## 22. Save the Error Analysis to Google Drive

Writes the classified failures to `RESULTS_DIR`. Notebook 02 reads this file as its starting point for the detailed error analysis.

In [22]:
# Failure rows with categories, consumed by notebook 02
failures.to_csv(
    RESULTS_DIR / "spider_1_5b_error_analysis.csv",
    index=False
)

print("Error analysis saved to:", RESULTS_DIR)

Error analysis saved to: /content/drive/MyDrive/Prajwal/Projects/SQLSmith/results
